In [14]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import pynbody
from pynbody.analysis import profile

import os
import glob

from tqdm.notebook import tqdm


from cosmoSim import cosmoSim

In [15]:
matplotlib.rc('xtick', labelsize=12)
matplotlib.rc('ytick', labelsize=12)
matplotlib.rcParams['font.size']=12

In [16]:
data_path = "/home/ryan/Data/"
base_path="/home/ryan/projects/Medvedev/data_product_generation/data_products/data_prods/"

out_path = '../plots/811_HY/'


try:
    os.mkdir(out_path)
except:
    print(f'{out_path} already exists!')

../plots/811_HY/ already exists!


In [17]:
# z_max = 5

# with open('../../file_parts/ExpansionList_128', 'r') as f:
#     lines = f.read().split('\n')

# a = np.array([ float(l.strip().split(' ')[0]) for l in lines if l])

# zs = 1/a - 1

# zs = zs[ zs < z_max ]

In [18]:
CDM_RUNS = [
    'run_CDM_811_HY_dir_0'
]

tcDM_search = f'{data_path}/run_2cDM_811_HY_power00*Vkick*'

tcDM_RUNS = [ os.path.basename(x) for x in sorted(glob.glob(tcDM_search)) ]

In [19]:
Vkicks = [ cosmoSim(x, base_path=base_path).Vkick for x in tcDM_RUNS ]

c_norm = matplotlib.colors.LogNorm(vmin=np.amin(Vkicks), vmax=np.amax(Vkicks))
c_map = matplotlib.cm.plasma

s_map = matplotlib.cm.ScalarMappable(cmap=c_map, norm=c_norm)

In [20]:
soft_eff = 100000 / 2**11 / 39

In [21]:
soft_eff

1.252003205128205

In [22]:
# def calculate_profile(run_name, redshift, halo_index=0):
    
#     run = cosmoSim(run_name, base_path=base_path)
#     idx = run.redshift_to_index(redshift)
#     # load the file
#     # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
#     f = pynbody.load(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
#     h = f.halos(subhalos=True)

#     f.physical_units()
#     f['rho']; f['smooth'] # force KDTree to be built and density estimations made
#     # only care about most massive halo for now
#     pynbody.analysis.faceon(h[halo_index])
#     min_rad = np.amin([ 0.1 * soft_eff, 0.1 * h[halo_index].properties['SubhaloHalfmassRad']])
#     max_rad = 3 * h[halo_index].properties['SubhaloHalfmassRad']
#     p_3d = profile.Profile(f, rmin = min_rad, rmax = max_rad, ndim = 3)
#     return p_3d['rbins'].in_units('kpc'), p_3d['density'].in_units('Msol kpc^-3'), h[halo_index].properties['SubhaloMassInRad']

In [23]:
def center_and_profile(f, h, halo_index):
    pynbody.analysis.center(h[halo_index])
    min_rad = np.amin([ 0.1 * soft_eff, 0.1 * h[halo_index].properties['SubhaloHalfmassRad']])
    max_rad = 3 * h[halo_index].properties['SubhaloHalfmassRad']
    p_3d = profile.Profile(f, rmin = min_rad, rmax = max_rad, ndim = 3)
    return p_3d['rbins'].in_units('kpc'), p_3d['density'].in_units('Msol kpc^-3'), h[halo_index].properties['SubhaloMassInRad']

In [ ]:
#ugly, hacky

packed = [plt.subplots(1, 1, figsize=[12, 12]) for i in range(3)]
figs = [ packed[i][0] for i in range(3) ]
axs = [ packed[i][1] for i in range(3) ]
redshift=0

y_mins = []
y_maxes = []

#load files
for rn in tqdm(tcDM_RUNS):
    run = cosmoSim(rn, base_path=base_path)
    idx = run.redshift_to_index(redshift)
    # load the file
    # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    f = pynbody.load(data_path + f'/{rn}/snap_{idx:03}.hdf5')
    h = f.halos(subhalos=True)

    f.physical_units()
    f['rho']; f['smooth'] # force KDTree to be built and density estimations made

    for halo_idx in range(3):
        # plot for all halos
        try:
            rbins, masses, _ = center_and_profile(f, h, halo_index=halo_idx)
        except KeyError:
            print(f'Error with run {rn} for profile {idx}')
            print('Skipping...')
            continue
        axs[halo_idx].plot(rbins, masses, color=s_map.to_rgba(run.Vkick))

        y_mins.append(1/2.8 * np.amin( masses[ np.nonzero(masses) ] ))
        y_maxes.append(2.8 * np.amax( masses[ np.nonzero(masses) ] ))
    del f

#same for CDM
for rn in CDM_RUNS:
    run = cosmoSim(rn, base_path=base_path)
    idx = run.redshift_to_index(redshift)
    # load the file
    # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    f = pynbody.load(data_path + f'/{rn}/snap_{idx:03}.hdf5')
    h = f.halos(subhalos=True)

    f.physical_units()
    f['rho']; f['smooth'] # force KDTree to be built and density estimations made
    for halo_idx in range(3):
        rbins, masses, halo_mass = center_and_profile(f, h, halo_index=halo_idx)
        axs[halo_idx].plot(rbins, masses, color='k', linestyle=(5, (10, 3)))

        # might as well do postprocessing here as well
        figs[halo_idx].colorbar(s_map, label='$V_{kick}$ [km s$^{-1}$]', ax=axs[halo_idx])
        # plot formatting
        axs[halo_idx].set_xlabel('$r$ [kpc]')
        axs[halo_idx].set_ylabel(r'$\rho$ [M$_{\odot}$ kpc$^{-3}$]')

        axs[halo_idx].vlines(soft_eff, 10**2, 10**50, colors='k', linestyles='dashed')
        axs[halo_idx].vlines(2.8*soft_eff, 10**2, 10**50, colors='k', linestyles='dashdot')

        y_mins.append(1/2.8 * np.amin( masses[ np.nonzero(masses) ] ))
        y_maxes.append(2.8 * np.amax( masses[ np.nonzero(masses) ] ))

        ylims = [
            np.amin(y_mins),
            np.amax(y_maxes)
        ]

        axs[halo_idx].set_ylim(ylims)

        axs[halo_idx].plot([],[], label=f'z = {redshift:.3f}', alpha=0)
        axs[halo_idx].plot([],[], label=f'Halo Mass = {float(halo_mass):.3e}', alpha=0)
        

        axs[halo_idx].set_yscale("log")
        axs[halo_idx].set_xscale("log")
        axs[halo_idx].grid(True, which="both", ls="-")
        axs[halo_idx].set_box_aspect(aspect=1)
        axs[halo_idx].legend()
        axs[halo_idx].set_title('(0, 0) Model')

        fname = f'halo{halo_idx}_811_HY_00_{idx:03}.png'
        figs[halo_idx].savefig(out_path + fname)

        plt.close(figs[halo_idx])
    del f

  0%|          | 0/9 [00:00<?, ?it/s]

In [25]:
CDM_RUNS = [
    'run_CDM_811_HY_dir_0'
]

tcDM_search = f'{data_path}/run_2cDM_811_HY_powerm2m2*Vkick*'

tcDM_RUNS = [ os.path.basename(x) for x in sorted(glob.glob(tcDM_search)) ]

Vkicks = [ cosmoSim(x, base_path=base_path).Vkick for x in tcDM_RUNS ]

c_norm = matplotlib.colors.LogNorm(vmin=np.amin(Vkicks), vmax=np.amax(Vkicks))
c_map = matplotlib.cm.plasma

s_map = matplotlib.cm.ScalarMappable(cmap=c_map, norm=c_norm)

In [26]:
#ugly, hacky

packed = [plt.subplots(1, 1, figsize=[12, 12]) for i in range(3)]
figs = [ packed[i][0] for i in range(3) ]
axs = [ packed[i][1] for i in range(3) ]
redshift=0

#load files
for rn in tqdm(tcDM_RUNS):
    run = cosmoSim(rn, base_path=base_path)
    idx = run.redshift_to_index(redshift)
    # load the file
    # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    f = pynbody.load(data_path + f'/{rn}/snap_{idx:03}.hdf5')
    h = f.halos(subhalos=True)

    f.physical_units()
    f['rho']; f['smooth'] # force KDTree to be built and density estimations made

    for halo_idx in range(3):
        # plot for all halos
        try:
            rbins, masses, _ = center_and_profile(f, h, halo_index=halo_idx)
        except KeyError:
            print(f'Error with run {rn} for profile {idx}')
            print('Skipping...')
            continue
        axs[halo_idx].plot(rbins, masses, color=s_map.to_rgba(run.Vkick))
    del f

#same for CDM
for rn in CDM_RUNS:
    run = cosmoSim(rn, base_path=base_path)
    idx = run.redshift_to_index(redshift)
    # load the file
    # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    f = pynbody.load(data_path + f'/{rn}/snap_{idx:03}.hdf5')
    h = f.halos(subhalos=True)

    f.physical_units()
    f['rho']; f['smooth'] # force KDTree to be built and density estimations made
    for halo_idx in range(3):
        rbins, masses, halo_mass = center_and_profile(f, h, halo_index=halo_idx)
        axs[halo_idx].plot(rbins, masses, color='k', linestyle=(5, (10, 3)))

        # might as well do postprocessing here as well
        figs[halo_idx].colorbar(s_map, label='$V_{kick}$ [km s$^{-1}$]', ax=axs[halo_idx])
        # plot formatting
        axs[halo_idx].set_xlabel('$r$ [kpc]')
        axs[halo_idx].set_ylabel(r'$\rho$ [M$_{\odot}$ kpc$^{-3}$]')

        axs[halo_idx].vlines(soft_eff, 10**2, 10**50, colors='k', linestyles='dashed')
        axs[halo_idx].vlines(2.8*soft_eff, 10**2, 10**50, colors='k', linestyles='dashdot')

        ylims = [
            1/2.8 * np.amin( masses[ np.nonzero(masses) ] ),
            2.8 * np.amax( masses[ np.nonzero(masses) ] )
        ]
        axs[halo_idx].set_ylim(ylims)

        axs[halo_idx].plot([],[], label=f'z = {redshift:.3f}', alpha=0)
        axs[halo_idx].plot([],[], label=f'Halo Mass = {float(halo_mass):.3e}', alpha=0)
        

        axs[halo_idx].set_yscale("log")
        axs[halo_idx].set_xscale("log")
        axs[halo_idx].grid(True, which="both", ls="-")
        axs[halo_idx].set_box_aspect(aspect=1)
        axs[halo_idx].legend()
        axs[halo_idx].set_title('(-2, -2) Model')

        fname = f'halo{halo_idx}_811_HY_mm_{idx:03}.png'
        figs[halo_idx].savefig(out_path + fname)

        plt.close(figs[halo_idx])
    del f

  0%|          | 0/7 [00:00<?, ?it/s]